# Agent CTF : essai gratuit (Kaggle ou Colab)

Ce notebook récupère l'agent depuis GitHub et le lance sur les challenges officiels CSAW 2026
(ou sur les challenges de démonstration). Deux façons d'avoir un modèle, toutes deux gratuites :

- **API Gemini gratuite** (par défaut) : modèle de pointe, quota quotidien. Il faut une clé gratuite.
- **Modèle ouvert local via Ollama** sur le GPU gratuit : aucune clé, mais modèle plus faible.

**Avant de lancer**
- Kaggle : *Settings* → *Internet* = **On** (compte vérifié par téléphone). Pour Ollama seulement :
  *Accelerator* = **GPU T4 x2**.
- Clé Gemini : la créer sur https://aistudio.google.com/apikey (gratuit, sans carte), puis
  Kaggle : *Add-ons* → *Secrets* → *Add secret*, nom **GEMINI_API_KEY**, et cocher la case du notebook.
  Colab : icône clé 🔑 à gauche, même nom, activer « Accès au notebook ».

Puis *Run all* (Kaggle) ou *Exécution* → *Tout exécuter* (Colab). Seule la cellule de paramètres se modifie.

In [ ]:
# --- Paramètres (seule cellule à modifier) -----------------------------------
# "gemini:gemini-3.8-flash" (API gratuite, clé GEMINI_API_KEY) ou "ollama:qwen2.5:32b" (GPU T4 x2)
MODEL = "gemini:gemini-3.8-flash"
BRANCH = "main"               # branche du dépôt à tester
# "csaw" = base officielle CSAW 2026 ; "demo" = 4 petits challenges de test du dépôt
CHALLENGES = "csaw"
CATEGORIES = []               # p. ex. ["crypto", "forensics"] ; vide = toutes
SKIP_SERVER = True            # pas de Docker sur Kaggle/Colab : on saute les challenges à serveur
SHELL = True                  # shell en bac à sable : sans réseau, sans accès à la base
MAX_STEPS = 40                # budget d'actions du LLM par challenge
REPO = "https://github.com/Chris-Lorys/LLM-CTF-Attack.git"
DB_REPO = "https://github.com/NYU-LLM-CTF/CSAW26_LLMAC_DB.git"

## 1. Préparer le modèle (clé d'API ou GPU)

In [ ]:
import os
USE_OLLAMA = MODEL.startswith("ollama:")
if USE_OLLAMA:
    !nvidia-smi --query-gpu=name,memory.total --format=csv
else:
    # La clé reste dans l'environnement du processus : jamais affichée, jamais écrite
    # dans la trace ni dans les résultats.
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("GEMINI_API_KEY")
        except Exception:
            try:
                from google.colab import userdata
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                key = None
    if not key:
        raise SystemExit("Clé GEMINI_API_KEY introuvable : voir « Avant de lancer » en haut du notebook.")
    os.environ["CSAW_API_KEY"] = key
    del key
    print("Clé d'API chargée ; Ollama ne sera pas installé.")

## 2. Installer et démarrer Ollama (seulement pour un modèle `ollama:`)

In [ ]:
if USE_OLLAMA:
    !apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
if USE_OLLAMA:
    import os, subprocess, time, urllib.request

    env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="16384", OLLAMA_KEEP_ALIVE="1h")
    server = subprocess.Popen(["ollama", "serve"], env=env,
                              stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
            print("Ollama est prêt.")
            break
        except Exception:
            time.sleep(1)
    else:
        print(open("/tmp/ollama.log").read()[-2000:])
        raise RuntimeError("Ollama n'a pas démarré : voir le journal ci-dessus")

In [ ]:
if USE_OLLAMA:
    OLLAMA_NAME = MODEL.split(":", 1)[1]
    !ollama pull {OLLAMA_NAME}

## 3. Récupérer l'agent et lancer les tests

In [ ]:
import os
# Kaggle a aussi un dossier /content et le module google.colab : on le reconnaît
# par sa variable d'environnement, sinon les résultats n'iraient pas dans Output.
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working")
os.chdir("/kaggle/working" if ON_KAGGLE else "/content")
!rm -rf LLM-CTF-Attack && git clone -q -b {BRANCH} {REPO}
os.chdir("LLM-CTF-Attack")
!git log --oneline -1
!pip -q install pytest
!python -m pytest -q

## 4. Lancer l'agent sur les challenges

### Outils du shell en bac à sable

Outils Linux mis à la disposition de l'agent, et utilisateur sans privilèges `ctfsandbox`
qui exécute ses commandes (sans réseau, sans accès à la base de challenges).

In [ ]:
if SHELL:
    !DEBIAN_FRONTEND=noninteractive apt-get -qq install -y file binutils p7zip-full unzip sqlite3 tshark xxd libimage-exiftool-perl sox > /dev/null
    !pip -q install pycryptodome scapy
    !id ctfsandbox 2>/dev/null || useradd -M -s /bin/bash ctfsandbox
    from csaw_agent.sandbox import Sandbox
    print("bac à sable :", Sandbox("ctfsandbox", []).describe())  # mode réseau retenu (root, userns ou seccomp)


In [ ]:
if CHALLENGES == "csaw":
    !rm -rf ../CSAW26_LLMAC_DB && git clone -q --depth 1 {DB_REPO} ../CSAW26_LLMAC_DB
    CHALLENGES_DIR = "../CSAW26_LLMAC_DB/dataset.json"
else:
    CHALLENGES_DIR = "tests/fixtures"
options = " ".join(f"--category {c}" for c in CATEGORIES) + (" --skip-server" if SKIP_SERVER else "")
if SHELL:
    # La base contient les solutions des organisateurs : l'agent la rend illisible
    # pour l'utilisateur du shell et vérifie qu'elle l'est avant de démarrer.
    options += " --shell"
!python -m csaw_agent list {CHALLENGES_DIR}
!python -m csaw_agent run-all {CHALLENGES_DIR} --model {MODEL} --max-steps {MAX_STEPS} --runs-dir runs {options}

## 5. Vérifier les traces et afficher un rapport

In [ ]:
import glob, json
from IPython.display import Markdown, display

for trace in sorted(glob.glob("runs/*/*/trace.jsonl")):
    print(trace)
    !python -m csaw_agent verify-trace {trace}

import os
if not os.path.exists("runs/bilan.json"):
    raise SystemExit("Pas de runs/bilan.json : le lancement de l'étape 4 a échoué, voir son message d'erreur.")
bilan = json.load(open("runs/bilan.json"))
for r in bilan:
    print(f"{r['status']:<20} {r.get('llm_steps') or 0:>3} action(s)  {r.get('id', r['challenge'])}")

premier = next((r for r in bilan if r.get("llm_steps")), None) or next(
    (r for r in bilan if r.get("report")), None)
if premier:
    display(Markdown(open(premier["report"]).read()))

## 6. Récupérer les résultats

L'archive contient, pour chaque challenge, la trace non retouchée (`trace.jsonl`), le rapport
(`rapport.md`) et le bilan global. Sur Kaggle, elle apparaît dans l'onglet *Output* ; sur Colab,
le téléchargement démarre tout seul.

In [ ]:
import os, shutil
archive = shutil.make_archive("../runs-" + MODEL.replace(":", "-").replace("/", "-"), "zip", "runs")
if ON_KAGGLE:
    # Seul /kaggle/working apparaît dans l'onglet Output.
    if os.path.dirname(os.path.abspath(archive)) != "/kaggle/working":
        archive = shutil.copy(archive, "/kaggle/working/")
    print(archive, "→ onglet Output (ou Data → /kaggle/working), ⋮ → Download")
else:
    print(archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:  # le navigateur bloque parfois le téléchargement automatique
        print(f"Téléchargement automatique impossible ({exc}).")
        print("Barre de gauche, icône dossier, /content, clic droit sur",
              archive.rsplit("/", 1)[-1], "puis Télécharger.")